# Estudo de caso 2.1 - Previsão de salários

Configuração do *notebook*:

Sincronize sua conta do Google. Para isso, siga o link que aparece na saída da seguinte célula uma vez executada. Copie o código que aparece na tela e insira-o na saída da célula. Assim que visualizar a mensagem: `Google Drive sincronizado com sucesso!` poderá continuar executando o restante das células.

In [ ]:
!pip install rpy2==3.5.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.7/201.7 kB 14.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for rpy2: filename=rpy2-3.5.1-cp310-cp310-linux_x86_64.whl size=318089 sha256=1ae37ea7c852262edf953ddf92faf88cb1e295176a60222ad3ad9679e9b579df
  Stored in directory: /root/.cache/pip/wheels/73/a6/ff/4e75dd1ce1cfa2b9a670cbccf6a1e41c553199e9b25f05d953
Successfully built rpy2
  Attempting uninstall: rpy2
    Found existing installation: rpy2 3.5.5
    Uninstalling rpy2-3.5.5:
      Successfully uninstalled rpy2-3.5.5


In [ ]:
from google.colab import auth
auth.authenticate_user()

from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)
data_drop = drive.CreateFile({'id':'1dnT7hcwXBklfjSHfiRjCGsLhL2aA_Feb'})
data_drop.GetContentFile('pay.discrimination.Rdata')

print('Google Drive sincronizado com sucesso!')

Google Drive sincronizado com sucesso!


In [ ]:
%load_ext rpy2.ipython

## Dados


In [ ]:
%%R
# Carregar o banco de dados
load(file="pay.discrimination.Rdata")

# Mostrar as variáveis do banco de dados
class(data)
str(data)

# Mostrar dimensões do banco de dados
dims  <- dim(data)
cat('\nDimensões do banco de dados:',toString(dims),'\n',fill = TRUE)

'data.frame':	3835 obs. of  12 variables:
 $ female: num  0 0 0 0 1 0 1 0 0 0 ...
 $ cg    : num  0 1 0 1 1 0 0 0 0 0 ...
 $ sc    : num  0 0 1 0 0 1 1 0 0 1 ...
 $ hsg   : num  1 0 0 0 0 0 0 1 1 0 ...
 $ mw    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ so    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ we    : num  0 0 0 0 0 0 0 0 0 0 ...
 $ ne    : num  1 1 1 1 1 1 1 1 1 1 ...
 $ exp1  : num  33 27 13 2 15 6.5 6 25 14 26 ...
 $ exp2  : num  10.89 7.29 1.69 0.04 2.25 ...
 $ exp3  : num  35.937 19.683 2.197 0.008 3.375 ...
 $ wage  : num  11.66 12.82 5.78 12.47 18.52 ...

Dimensões do banco de dados: 3835, 12 



Elaboração de uma tabela com a média de cada variável:

In [ ]:
%%R
# Tabela com a média de cada variável
stats <- as.matrix(apply(data, 2, mean))

# Atribuir o nome da coluna a status
colnames(stats) = "average"
print(stats,digits=2)

       average
female    0.42
cg        0.38
sc        0.32
hsg       0.30
mw        0.29
so        0.24
we        0.21
ne        0.26
exp1     13.35
exp2      2.53
exp3      5.81
wage     15.53


## Metodologia

### Sem separação de dados

Modelo básico:

In [ ]:
%%R
# Regressão linear do salário
fmla1     <-  wage ~ female + sc + cg + mw + so + we + exp1 + exp2 + exp3

# Execução da especificação linear e cálculo do EQM e R^2
full.fit1 <-  lm(fmla1, data=data)
fit1      <-  summary(full.fit1)
R2.1      <-  fit1$r.squared
R2.adj1   <-  fit1$adj.r.squared
n1        <-  length(fit1$res)
p1        <-  fit1$df[1]
MSE.adj1  <-  (n1/(n1-p1))*mean(fit1$res^2)

Modelo flexível:

In [ ]:
%%R
# Regressão linear: especificação quadrática
# O operador ^ no objeto fórmula específica na ordem das interações
fmla2     <- wage ~  female + (sc+ cg+ mw + so + we + exp1 + exp2 + exp3)^2

# Execução da especificação quadrática e cálculo do EQM e R^2
full.fit2 <- lm(fmla2, data=data)
fit2      <- summary(full.fit2)
R2.2      <- fit2$r.squared
R2.adj2   <- fit2$adj.r.squared
n2        <- length(fit2$res)
p2        <- fit2$df[1]
MSE.adj2  <- (n2/(n2-p2))*mean(fit2$res^2)

In [ ]:
%%R
# Resumo das especificações linear e quadrática
table1     <- matrix(0, 2, 4)
table1[1,] <- c(p1, R2.1, R2.adj1, MSE.adj1)
table1[2,] <- c(p2, R2.2, R2.adj2, MSE.adj2)

# Atribuir nomes de filas e colunas
colnames(table1) <- c("p", "R^2 amostra", "R^2 ajustado", "EQM ajustado")
rownames(table1) <- c("reg básica", "reg flexível")

### Com separação de dados

Modelo básico:

In [ ]:
%%R
# Estabelecendo o gerador de números aleatórios
set.seed(1)

# Separação dos dados nos conjuntos de treinamento e teste
train      <- sample(1:nrow(data), nrow(data)/2)

# Execução da especificação linear e cálculo do EQM e R^2 com a amostra de teste
full.fit1  <- lm(fmla1, data=data[train,])
yhat.fit1  <- predict(full.fit1, newdata=data[-train,])
y.test     <- data[-train,]$wage
MSE.fit1   <- summary(lm((y.test-yhat.fit1)^2~1))$coef[1]
R2.fit1    <- 1- MSE.fit1/var(y.test)

Modelo flexível:

In [ ]:
%%R
# Separação dos dados nos conjuntos de treinamento e teste
train      <- sample(1:nrow(data), nrow(data)/2)

# Execução da especificação quadrática e cálculo do EMC e R^2 com a amostra de teste
full.fit2  <- lm(fmla2, data=data[train,])
yhat.fit2  <- predict(full.fit2, newdata=data[-train,])
y.test     <- data[-train,]$wage
MSE.fit2   <- summary(lm((y.test-yhat.fit2)^2~1))$coef[1]
R2.fit2    <- 1- MSE.fit2/var(y.test)

In [ ]:
%%R
# Elaboração da tabela de resultados
table2      <- matrix(0, 2, 3)
table2[1,]  <- c(p1, R2.fit1, MSE.fit1)
table2[2,]  <- c(p2, R2.fit2, MSE.fit2)

# Atribuir nomes de filas e colunas
colnames(table2)  <- c("p ", "R^2 teste", "EQM teste")
rownames(table2)  <- c("reg básica", "reg flexível")

## Resultados

In [ ]:
%%R
# Mostrando os resultados
cat('- Resultados sem separação de dados:\n',fill = TRUE)
print(table1,digits=4)

cat('\n\n- Resultados com separação de dados:\n',fill = TRUE)
print(table2,digits=4)

- Resultados sem separação de dados:

              p R^2 amostra R^2 ajustado EQM ajustado
reg básica   10     0.09549      0.09336        165.7
reg flexível 33     0.10397      0.09643        165.1


- Resultados com separação de dados:

             p  R^2 teste EQM teste
reg básica   10   0.08135     224.1
reg flexível 33   0.09058     198.4
